# Explore bat records near mineral locations

This notebook reads the reproducible output generated from the supplied bat point shapefile, the 2006 USGS facility table, the 2017 USGS critical-mineral point layer, and a separately attributed Kibali location. It reports spatial proximity only; the bat source does not identify confirmed roosts.

In [ ]:
import gzip
import json
from pathlib import Path

candidates = [
    Path('../../data/storymap.json.gz'),
    Path('../data/storymap.json.gz'),
    Path('data/storymap.json.gz'),
    Path('../data/storymap.json'),
    Path('../_static/data/storymap.json'),
    Path('data/storymap.json'),
    Path('_static/data/storymap.json'),
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError('Build the project data first with `python scripts/build_data.py`.')
if data_path.suffix == '.gz':
    with gzip.open(data_path, 'rt', encoding='utf-8') as data_file:
        story = json.load(data_file)
else:
    story = json.loads(data_path.read_text(encoding='utf-8'))
story['summary']

## Locations with nearby records

Each row below is a mine classified as such in the 2006 USGS inventory, a 2017 USGS critical-mineral resource feature (the source does not specify whether a point is a mine, deposit, district, or region), or the supplemental Kibali point, with bat records at or inside 50 km. Buffer distances are ellipsoidal geodesic distances from the selected location.

In [ ]:
for page in story['pages']:
    facility = page['facility']
    print(
        f"{facility['name']} ({facility['country']}; {facility['source_label']}) — "
        f"{page['nearby_record_count']} records; nearest "
        f"{page['nearest_record_km']:.2f} km"
    )
    print('  distance bands:', page['band_counts'])

## Species represented in a location chapter

Select another item in `story['pages']` to inspect its species and source localities. The coordinates stored in each source feature retain the original valid source precision.

In [ ]:
if story['pages']:
    page = story['pages'][0]
    for record in page['bat_records']:
        print(
            f"{record['scientific_name']} ({record['english_name'] or 'common name not verified'}) | "
            f"{record['location']}, {record['country']} | "
            f"{record['distance_km']:.2f} km | {record['reference']}"
        )
else:
    print('No coordinate-valid location has a point record within 50 km in these sources.')